### Transform Results Data

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read the bronze Results table

In [0]:
results_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))

#### Step 2 - Drop URL column

In [0]:
results_selected_df = results_df.select(
    "season",
    "round",
    "constructorId",
    "driverId",
    "date",
    "raceName",
    "grid",
    "laps",
    "number",
    "points",
    "position",
    "positionText",
    "status",
    "ingestion_timestamp",
    "source",
    "batch_id"
)

#### Step 3 & 4 - Standardizing columns by snake case and readability

In [0]:
results_renamed_df = results_selected_df.withColumnsRenamed({
    "constructorId": "constructor_id",
    "driverId": "driver_id",
    "raceName": "race_name",
    "date": "race_date",
    "grid": "grid_position",
    "laps": "completed_laps",
    "number": "car_number",
    "position": "final_position",
    "positionText": "final_position_text"
    })

#### Step 5 - Check validity(NULL Handling)

In [0]:
results_valid_df = results_renamed_df.filter(
    F.col("constructor_id").isNotNull() &
    F.col("driver_id").isNotNull() & 
    F.col("season").isNotNull() &
    F.col("round").isNotNull()
)

In [0]:
display(results_valid_df.count() - results_renamed_df.count())

#### Step 6 - Remove duplicates

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season","round","constructor_id", "driver_id"])

In [0]:
display(results_valid_df.count() - results_distinct_df.count())

#### Step 7 - Change the values in race_name to Title Case

In [0]:
results_final_df = results_distinct_df.withColumn("race_name", F.initcap(F.col("race_name")))

#### Step 8 - Write the data into races silver table

In [0]:
write_to_silver(
    input_df=results_final_df,
    target_table=silver_table,
    merge_condition="s.season = t.season AND s.round = t.round AND s.constructor_id = t.constructor_id AND s.driver_id = t.driver_id",
    columns_to_update=[
        "race_date",
        "race_name",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source",
        "batch_id"
    ]
)